## LeNet5

Outputs

- `models/lenet5.tflite` normal (INT8 input/output, per-tensor; MicroFlow-compatible)
- `models/lenet5_quantized.tflite` quantized (INT8 input/output, per-tensor; MicroFlow-compatible)

Keep `epochs` small if you only need artifacts quickly.

In [6]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE = MODELS_DIR / "lenet5.tflite"
OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized.tflite"

print("OUT_TFLITE:", OUT_TFLITE)
print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)

OUT_TFLITE: /home/nathan/Documents/ariel-microflow-ml/models/lenet5.tflite
OUT_TFLITE_QUANT: /home/nathan/Documents/ariel-microflow-ml/models/lenet5_quantized.tflite


In [7]:
def build_lenet5() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(28, 28, 1), batch_size=1, name="input")
    x = tf.keras.layers.Conv2D(6, (5, 5), activation="relu", padding="valid")(inputs)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Conv2D(16, (5, 5), activation="relu", padding="valid")(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = tf.keras.layers.Reshape((256,), name="flatten_256")(x)
    x = tf.keras.layers.Dense(120, activation="relu", name="fc1")(x)
    x = tf.keras.layers.Dense(84, activation="relu", name="fc2")(x)
    logits = tf.keras.layers.Dense(10, name="fc3")(x)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="lenet5")

model = build_lenet5()
model.summary()

Model: "lenet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 28, 28, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (1, 24, 24, 6)         │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (1, 12, 12, 6)         │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (1, 8, 8, 16)          │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (1, 4, 4, 16)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_256 (Reshape)           │ (1, 256)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (1, 120)               │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (1, 84)                │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (1, 10)                │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:

epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype(np.float32) / 255.0)[..., None]
x_test = (x_test.astype(np.float32) / 255.0)[..., None]

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2
422/422 - 5s - 13ms/step - accuracy: 0.8828 - loss: 0.4033 - val_accuracy: 0.9635 - val_loss: 0.1218
Epoch 2/2
422/422 - 2s - 6ms/step - accuracy: 0.9654 - loss: 0.1136 - val_accuracy: 0.9748 - val_loss: 0.0845
Test accuracy: 0.9753999710083008


In [9]:
NORMAL_REP_SAMPLES = 50

def representative_data_gen_normal():
    n = min(NORMAL_REP_SAMPLES, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

float_converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_bytes = float_converter.convert()
OUT_TFLITE.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE, "bytes=", OUT_TFLITE.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE), experimental_delegates=[])
interp.allocate_tensors()
print("in ", interp.get_input_details()[0]["dtype"], interp.get_input_details()[0]["shape"])
print("out ", interp.get_output_details()[0]["dtype"], interp.get_output_details()[0]["shape"])

per_channel = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK")

try:
    ops = interp._get_ops_details()
    print("Ops:", [op.get("op_name") for op in ops])
except Exception as e:
    print("Couldn't list ops:", e)


Saved artifact at '/tmp/tmpwnsquypz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  138826836998864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826837000016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836997520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836996752: TensorSpec(shape=(), dtype=tf.resource, name=None)
Wrote: /home/nathan/Documents/

W0000 00:00:1774868176.898447   46462 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774868176.898462   46462 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-30 12:56:16.898623: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpwnsquypz
2026-03-30 12:56:16.899212: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-30 12:56:16.899233: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpwnsquypz
2026-03-30 12:56:16.902806: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-30 12:56:16.924796: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpwnsquypz
2026-03-30 12:56:16.931560: I tensorflow/cc/saved_model/loader.cc:466] SavedModel load for tags { serve }; Status: success: OK. Took 32943 microseconds.
2026-03-30 12:56:16.971434: I tensorflow/comp

In [10]:
def representative_data_gen():
    for i in range(200):
        yield [x_train[i : i + 1].astype(np.float32)]

int8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_data_gen
int8_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_converter.target_spec.supported_types = [tf.int8]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

for attr in (
    "_experimental_disable_per_channel",
    "_experimental_disable_per_channel_quantization",
):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, True)

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(int8_converter, attr):
        setattr(int8_converter, attr, False)

int8_tflite = int8_converter.convert()
OUT_TFLITE_QUANT.write_bytes(int8_tflite)

int8_interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
int8_interp.allocate_tensors()
in_info = int8_interp.get_input_details()[0]
out_info = int8_interp.get_output_details()[0]

print("wrote", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)
print("int8 input:", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("int8 output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])

per_channel = []
for td in int8_interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK (all tensors per-tensor or unquantized)")


Saved artifact at '/tmp/tmpq5fua8e9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 28, 28, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  138826836998864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826837000016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836998480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836997520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836999248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138826836996752: TensorSpec(shape=(), dtype=tf.resource, name=None)
wrote /home/nathan/Documents/a

/home/nathan/Documents/ariel-microflow-ml/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774868177.218579   46462 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774868177.218592   46462 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-30 12:56:17.218732: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpq5fua8e9
2026-03-30 12:56:17.219339: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-30 12:56:17.219356: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpq5fua8e9
2026-03-30 12:56:17.223672: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-30 12:56:17.249630: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel 